### EL CODIGO DEL PREPROCESADO TRADUCIDO A SPARK
#### NO USA LIBRERIAS PROPIAS (PARA PODER EJECUTARLO EN EL CLUSTER)

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, lit, length
from pyspark import keyword_only
from pyspark.sql.types import StringType, IntegerType, DoubleType, LongType
import pyModeS as pms
import pyModeS.decoder.bds.bds60 as bds60
import base64
from pyspark.ml import Pipeline, Transformer

#### Definimos las funciones y clase (antes esto se ubicaba dentro de las librerías)

In [2]:
def base64toHEX(b64):
    return base64.b64decode(b64).hex()

def getDownlink(hex):
    return pms.df(hex)

def getTypeCode(hex):
    return pms.common.typecode(hex)

def getICAO(hex):
    return str(pms.common.icao(hex))

def msgIsCorrupted(hex):
    return (pms.crc(hex) != 0)

def getOnGround(hex):
    decimal_value =  pms.bin2int(pms.hex2bin(hex)[5:8]) 
    if decimal_value == 4:
        return 1
    elif decimal_value == 5:
        return 0
    else:
        return None

In [3]:
class AircraftIdentificationMessage():

    def __init__(self):
        # DICCIONARIO DE COMBINACIONES PARA VARIABLE VORTEX,PRIMER NIVEL CAT, SEGUNDO NIVEL TC
        self.vortexDictionary = {
            1: {
                0: "No category information",
                1: "Reserved",
                2: "Reserved",
                3: "Reserved",
                4: "Reserved",
                5: "Reserved",
                6: "Reserved",
                7: "Reserved"
            },
            2: {
                0: "No category information",
                1: "Surface emergency vehicle",
                2: "ERROR",
                3: "Surface service vehicle",
                4: "Ground obstruction",
                5: "Ground obstruction",
                6: "Ground obstruction",
                7: "Ground obstruction"
            },
            3: {
                0: "No category information",
                1: "Glider, sailplane",
                2: "Lighter-than-air",
                3: "Parachutist, skydiver",
                4: "Ultralight, hang-glider, paraglider",
                5: "Reserved",
                6: "Unmanned aerial vehicle",
                7: "Space or transatmospheric vehicle"
            },
            4: {
                0: "No category information",
                1: "Light (less than 7000 kg)",
                2: "Medium 1 (between 7000 kg and 34000 kg)",
                3: "Medium 2 (between 34000 kg to 136000 kg)",
                4: "High vortex aircraft",
                5: "Heavy (larger than 136000 kg)",
                6: "High performance (>5 g acceleration) and high speed (>400 kt)",
                7: "Rotorcraft"
            }
        }

    def match(self, typecode):
        return typecode >= 1 and typecode <= 4
    

    def getCA(self, hex):
        tc = self.getTypeCode(hex)
        # Asegurar que tc no es None antes de comparar
        if tc is None or not self.match(tc):
            return None
        try:
            return pms.decoder.adsb.category(hex)
        except Exception:
            return None


    def getAircraftType(self, hex):
        tc = self.getTypeCode(hex)
        ca = self.getCA(hex)

        if tc in self.vortexDictionary and ca in self.vortexDictionary[tc]:
            return self.vortexDictionary[tc][ca]
        return None
    
    def updateRowFromHex(self, row, hex):
        callsign = pms.decoder.adsb.callsign(hex)
        tc = self.getTypeCode(hex)
        cat = pms.decoder.adsb.category(hex)
        row["callsign"] = callsign
        row["vortex"] = self.vortexDictionary[tc][cat]

    
    def getTypeCode(self, hex):
        return pms.common.typecode(hex)

## CODIGO PRINCIPAL

### PARTE 1

In [4]:
# Wrappers para que sean funciones spark (User Defined Function)

base64toHEX_udf = udf(base64toHEX,StringType())
getDownlink_udf = udf(getDownlink, IntegerType())
getICAO_udf = udf(getICAO, StringType())
getOnGround_udf = udf(getOnGround, IntegerType())
getTypeCode_udf = udf(getTypeCode, IntegerType())
getHeading_udf = udf(bds60.hdg60, DoubleType())
getVerticalRate_udf = udf(bds60.vr60ins, IntegerType())
getAircraftType_udf = udf(AircraftIdentificationMessage().getAircraftType, StringType())

In [5]:
class ColumnDecoder(Transformer):
    @keyword_only
    def __init__(self, column_name=None, value=None):
        super().__init__()

    def _transform(self, df):
        # add your transformation logic here
        df = df\
            .withColumn("messageHex", base64toHEX_udf(col("message")))\
            .withColumn("DL", getDownlink_udf(col("messageHex")))\
            .withColumn("ICAO", getICAO_udf(col("messageHex")))\
            .withColumn("timestamp", (col("ts_kafka") / 1000).cast("timestamp"))\
            .withColumn("messageLen", length(col("messageHex")))

        df_17_18 = df.filter(col("DL").isin([17, 18]) & (col("messageLen") == 28))
        df_20_21 = df.filter(col("DL").isin([20, 21]) & (col("messageLen") == 28))


        df_17_18 = df_17_18\
            .withColumn("OnGround", getOnGround_udf(col("messageHex")))\
            .withColumn("TC", getTypeCode_udf(col("messageHex")))\
            .withColumn("AircraftType", getAircraftType_udf(col("messageHex")))

        df_20_21 = df_20_21\
            .withColumn("heading", getHeading_udf(col("messageHex")))\
            .withColumn("vertical_rate", getVerticalRate_udf(col("messageHex")))

        ## Añadimos columnas vacías para poder juntar los dfs
        df_17_18 = df_17_18\
            .withColumn('heading', lit(None).cast(DoubleType()))\
            .withColumn('vertical_rate', lit(None).cast(DoubleType()))

        df_20_21 = df_20_21\
            .withColumn('OnGround', lit(None).cast(IntegerType()))\
            .withColumn('TC', lit(None).cast(IntegerType()))\
            .withColumn('AircraftType', lit(None).cast(StringType()))

        ## Juntamos dataframes y ordenamos por timestamp para volver al orden anterior
        df_merged_ini = df_17_18.unionByName(df_20_21).orderBy(col("timestamp").asc())
        return df_merged_ini

### PARTE 2

### Tests

In [6]:
spark = SparkSession.builder.appName("procesadillo").getOrCreate()
#spark.sparkContext.setLogLevel("ERROR")
path = "../../../test.csv"

df = spark.read.options(delimiter=";", header=True).csv(path)
df = df.withColumn("ts_kafka",col("ts_kafka").cast(LongType()))
df = df.drop("_c2") # se llama diferente en pyspark
print(df.dtypes)

df_decoded = ColumnDecoder().transform(df)
df_decoded.show(10)

#df_decoded.repartition(1).write.mode("overwrite").csv("../../../datos_prueba_sp.csv", header=True)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/01 12:27:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


[('ts_kafka', 'bigint'), ('message', 'string')]


+-------------+--------------------+--------------------+---+------+--------------------+----------+--------+----+--------------------+------------+-------------+
|     ts_kafka|             message|          messageHex| DL|  ICAO|           timestamp|messageLen|OnGround|  TC|        AircraftType|     heading|vertical_rate|
+-------------+--------------------+--------------------+---+------+--------------------+----------+--------+----+--------------------+------------+-------------+
|1734667201534|kDQjSSgABjG4hW4fTbY=|9034234928000631b...| 18|342349|2024-12-20 05:00:...|        28|    NULL|   5|                NULL|        NULL|         NULL|
|1734667201534|oAANuYAVzS4ABLg3aB0=|a0000db98015cd2e0...| 20|AA5F41|2024-12-20 05:00:...|        28|    NULL|NULL|                NULL|  0.17578125|       5888.0|
|1734667201534|kDQjShBRgxXDaCAWeDM=|9034234a10518315c...| 18|34234a|2024-12-20 05:00:...|        28|    NULL|   2|No category infor...|        NULL|         NULL|
|1734667201534|oAAOn4u

25/04/01 12:27:38 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
